<a href="https://colab.research.google.com/github/haqiqien/LoRA/blob/main/LoRA-default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Fine-Tune LLMs with LoRA Adapters using Hugging Face TRL

This notebook demonstrates how to efficiently fine-tune large language models using LoRA (Low-Rank Adaptation) adapters. LoRA is a parameter-efficient fine-tuning technique that:
- Freezes the pre-trained model weights
- Adds small trainable rank decomposition matrices to attention layers
- Typically reduces trainable parameters by ~90%
- Maintains model performance while being memory efficient

We'll cover:
1. Setup development environment and LoRA configuration
2. Create and prepare the dataset for adapter training
3. Fine-tune using `trl` and `SFTTrainer` with LoRA adapters
4. Test the model and merge adapters (optional)


## 1. Setup development environment

Our first step is to install Hugging Face Libraries and Pytorch, including trl, transformers and datasets. If you haven't heard of trl yet, don't worry. It is a new library on top of transformers and datasets, which makes it easier to fine-tune, rlhf, align open LLMs.


In [1]:
# Install the requirements in Google Colab
%pip install -q -U transformers datasets trl peft accelerate huggingface_hub
%pip install -q --upgrade --force-reinstall --no-cache-dir "torchao>=0.16.0"

# Authenticate to Hugging Face

from huggingface_hub import login

login()

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.0 MB/s eta 0:00:00


## 2. Load the dataset

In [2]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
dataset = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
dataset

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/everyday-conversations/train-00000-(…): reconstructing file:   0%|          |  0.00B /  946kB            

data/everyday-conversations/train-00000-(…): downloading bytes:           |  0.00B            

data/everyday-conversations/test-00000-o(…): reconstructing file:   0%|          |  0.00B / 52.6kB            

data/everyday-conversations/test-00000-o(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2260 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/119 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 2260
    })
    test: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 119
    })
})

## 3. Fine-tune LLM using `trl` and the `SFTTrainer` with LoRA

The [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) from `trl` provides integration with LoRA adapters through the [PEFT](https://huggingface.co/docs/peft/en/index) library. Key advantages of this setup include:

1. **Memory Efficiency**:
   - Only adapter parameters are stored in GPU memory
   - Base model weights remain frozen and can be loaded in lower precision
   - Enables fine-tuning of large models on consumer GPUs

2. **Training Features**:
   - Native PEFT/LoRA integration with minimal setup
   - Support for QLoRA (Quantized LoRA) for even better memory efficiency

3. **Adapter Management**:
   - Adapter weight saving during checkpoints
   - Features to merge adapters back into base model

We'll use LoRA in our example, which combines LoRA with 4-bit quantization to further reduce memory usage without sacrificing performance. The setup requires just a few configuration steps:
1. Define the LoRA configuration (rank, alpha, dropout)
2. Create the SFTTrainer with PEFT config
3. Train and save the adapter weights


In [3]:
# Import necessary libraries
%pip install -q -U transformers datasets trl peft accelerate huggingface_hub
%pip install -q --upgrade --force-reinstall --no-cache-dir "torchao>=0.16.0"

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Provide a fallback chat template when the downloaded tokenizer does not
# include one (required for conversational datasets in SFTTrainer).
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{ '<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>\\n' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\\n' }}{% endif %}"
    )

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset"
finetune_tags = ["smol-course", "module_1"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 58.2 MB/s eta 0:00:00


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

The `SFTTrainer`  supports a native integration with `peft`, which makes it super easy to efficiently tune LLMs using, e.g. LoRA. We only need to create our `LoraConfig` and provide it to the trainer.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Define LoRA parameters for finetuning</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p>
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the general parameters for an abitrary finetune</p>
    <p>🐕 Adjust the parameters and review in weights & biases.</p>
    <p>🦁 Adjust the parameters and show change in inference results.</p>
</div>

In [4]:
from peft import LoraConfig

# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

Before we can start our training we need to define the hyperparameters (`TrainingArguments`) we want to use.

In [5]:
# Training configuration
# Hyperparameters based on QLoRA paper recommendations
# TRL versions differ: older versions accept warmup_ratio, while newer
# versions expose warmup_steps instead.
import inspect
import math

if "warmup_ratio" in inspect.signature(SFTConfig).parameters:
    warmup_kwargs = {"warmup_ratio": 0.03}
else:
    updates_per_epoch = math.ceil(len(dataset["train"]) / (2 * 2))
    warmup_kwargs = {"warmup_steps": max(1, round(0.03 * updates_per_epoch))}

args = SFTConfig(
    # Output settings
    output_dir=finetune_name,  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=1,  # Number of training epochs
    # Batch size settings
    per_device_train_batch_size=2,  # Batch size per GPU
    gradient_accumulation_steps=2,  # Accumulate gradients for larger effective batch
    # Memory optimization
    gradient_checkpointing=True,  # Trade compute for memory savings
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-4,  # Learning rate (QLoRA paper)
    max_grad_norm=0.3,  # Gradient clipping threshold
    # Learning rate schedule
    **warmup_kwargs,  # 3% warmup, compatible with the installed TRL version
    lr_scheduler_type="constant",  # Keep learning rate constant after warmup
    # Logging and saving
    logging_steps=1,  # Log metrics every training step
    disable_tqdm=False,  # Show the training progress bar
    save_strategy="epoch",  # Save checkpoint every epoch
    # Precision settings
    # Enable bf16 only on supported CUDA hardware; use fp32 otherwise.
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to="none",  # Disable external logging
)

We now have every building block we need to create our `SFTTrainer` to start then training our model.

In [6]:
max_seq_length = 1512  # max sequence length for model and packing of the dataset

# Define separate LoRA experiments. Each run changes one or more LoRA parameters.
experiment_configs = [
    {"name": "r4_a8_d005_all", "r": 4, "alpha": 8, "dropout": 0.05, "target_modules": "all-linear"},
    {"name": "r8_a16_d005_all", "r": 8, "alpha": 16, "dropout": 0.05, "target_modules": "all-linear"},
    {"name": "r8_a16_d010_qv", "r": 8, "alpha": 16, "dropout": 0.10, "target_modules": ["q_proj", "v_proj"]},
    {"name": "r16_a32_d010_qv", "r": 16, "alpha": 32, "dropout": 0.10, "target_modules": ["q_proj", "v_proj"]},
]

def create_experiment_trainer(run_model, run_args, config):
    run_peft_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["alpha"],
        lora_dropout=config["dropout"],
        bias="none",
        target_modules=config["target_modules"],
        task_type="CAUSAL_LM",
    )
    trainer_parameters = inspect.signature(SFTTrainer).parameters
    trainer_kwargs = {
        "model": run_model,
        "args": run_args,
        "train_dataset": dataset["train"],
        "peft_config": run_peft_config,
    }
    if "packing" in trainer_parameters:
        trainer_kwargs["packing"] = True
    elif hasattr(run_args, "packing"):
        run_args.packing = True
    if "dataset_kwargs" in trainer_parameters:
        trainer_kwargs["dataset_kwargs"] = {
            "add_special_tokens": False,
            "append_concat_token": False,
        }
    if "max_seq_length" in trainer_parameters:
        trainer_kwargs["max_seq_length"] = max_seq_length
    elif "max_length" in trainer_parameters:
        trainer_kwargs["max_length"] = max_seq_length
    elif hasattr(run_args, "max_length"):
        run_args.max_length = max_seq_length
    elif hasattr(run_args, "max_seq_length"):
        run_args.max_seq_length = max_seq_length
    if "tokenizer" in trainer_parameters:
        trainer_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in trainer_parameters:
        trainer_kwargs["processing_class"] = tokenizer
    return SFTTrainer(**trainer_kwargs)

Tokenizing train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Start training our model by calling the `train()` method on our `Trainer` instance. This will start the training loop and train our model for 3 epochs. Since we are using a PEFT method, we will only save the adapted model weights and not the full model.

In [7]:
# Run each LoRA configuration and compare the results.
from copy import deepcopy
import gc
import pandas as pd
from transformers import TrainerCallback
import time

class ConsoleProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            self.last_metrics = logs

    def on_step_end(self, args, state, control, **kwargs):
        if state.max_steps and state.global_step > 0:
            elapsed = time.time() - self.start_time
            speed = state.global_step / max(elapsed, 1e-6)
            eta = (state.max_steps - state.global_step) / max(speed, 1e-6)
            metrics = getattr(self, "last_metrics", {})
            print(
                f"Progress {100 * state.global_step / state.max_steps:6.2f}% | "
                f"step {state.global_step}/{state.max_steps} | epoch {state.epoch:.2f} | "
                f"loss {metrics.get('loss', '-')} | ETA {eta / 60:.1f} min",
                flush=True,
            )

experiment_results = []
# Free the single-model object created in the setup cell before starting runs.
if "model" in globals():
    del model
    gc.collect()

for run_number, config in enumerate(experiment_configs, start=1):
    run_name = config["name"]
    run_output_dir = f"{finetune_name}-exp-{run_name}"
    run_args = deepcopy(args)
    run_args.output_dir = run_output_dir
    print(f"\n===== RUN {run_number}/{len(experiment_configs)}: {run_name} =====", flush=True)
    print(f"Configuration: {config}", flush=True)
    run_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    trainer = create_experiment_trainer(run_model, run_args, config)
    trainer.add_callback(ConsoleProgressCallback())
    start_time = time.time()
    train_result = trainer.train()
    trainer.save_model(run_output_dir)
    result = {**config, "output_dir": run_output_dir, **train_result.metrics}
    result["wall_time_minutes"] = round((time.time() - start_time) / 60, 2)
    experiment_results.append(result)
    print(f"Finished {run_name}: {result}", flush=True)
    del trainer, run_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_table = pd.DataFrame(experiment_results).sort_values("train_loss", na_position="last")
display(results_table)
results_table.to_csv("lora_experiment_results.csv", index=False)
print("Results saved to lora_experiment_results.csv")

Starting training: 1 epoch(s), 2260 training examples


Step,Training Loss
1,2.740185
2,2.595839
3,2.632678
4,2.664981
5,2.587219
6,2.581766
7,2.595881
8,2.709640
9,2.466529
10,2.482975


Progress:   1.33% | step 1/75 | epoch 0.01 | loss - | lr - | elapsed 0.1 min | ETA 9.5 min
Progress:   2.67% | step 2/75 | epoch 0.03 | loss 2.740184783935547 | lr 0.0002 | elapsed 0.2 min | ETA 8.5 min
Progress:   4.00% | step 3/75 | epoch 0.04 | loss 2.595839023590088 | lr 0.0002 | elapsed 0.3 min | ETA 8.4 min
Progress:   5.33% | step 4/75 | epoch 0.05 | loss 2.6326775550842285 | lr 0.0002 | elapsed 0.5 min | ETA 8.1 min
Progress:   6.67% | step 5/75 | epoch 0.07 | loss 2.664980888366699 | lr 0.0002 | elapsed 0.6 min | ETA 8.0 min
Progress:   8.00% | step 6/75 | epoch 0.08 | loss 2.58721923828125 | lr 0.0002 | elapsed 0.7 min | ETA 7.9 min
Progress:   9.33% | step 7/75 | epoch 0.09 | loss 2.581766128540039 | lr 0.0002 | elapsed 0.8 min | ETA 7.8 min
Progress:  10.67% | step 8/75 | epoch 0.11 | loss 2.5958809852600098 | lr 0.0002 | elapsed 0.9 min | ETA 7.6 min
Progress:  12.00% | step 9/75 | epoch 0.12 | loss 2.7096400260925293 | lr 0.0002 | elapsed 1.0 min | ETA 7.3 min
Progress:  

[{'loss': 2.740184783935547,
  'grad_norm': 0.45671379566192627,
  'learning_rate': 0.0002,
  'entropy': 2.648897409439087,
  'num_tokens': 5966.0,
  'mean_token_accuracy': 0.5106429755687714,
  'epoch': 0.013422818791946308,
  'step': 1},
 {'loss': 2.595839023590088,
  'grad_norm': 0.4208783209323883,
  'learning_rate': 0.0002,
  'entropy': 2.5145413875579834,
  'num_tokens': 11680.0,
  'mean_token_accuracy': 0.527500331401825,
  'epoch': 0.026845637583892617,
  'step': 2},
 {'loss': 2.6326775550842285,
  'grad_norm': 0.5100688934326172,
  'learning_rate': 0.0002,
  'entropy': 2.5257073640823364,
  'num_tokens': 17650.0,
  'mean_token_accuracy': 0.5252607464790344,
  'epoch': 0.040268456375838924,
  'step': 3},
 {'loss': 2.664980888366699,
  'grad_norm': 0.47647884488105774,
  'learning_rate': 0.0002,
  'entropy': 2.5755292177200317,
  'num_tokens': 23557.0,
  'mean_token_accuracy': 0.5087500214576721,
  'epoch': 0.053691275167785234,
  'step': 4},
 {'loss': 2.58721923828125,
  'grad_

The training with Flash Attention for 3 epochs with a dataset of 15k samples took 4:14:36 on a `g5.2xlarge`. The instance costs `1.21$/h` which brings us to a total cost of only ~`5.3$`.



### Merge LoRA Adapter into the Original Model

When using LoRA, we only train adapter weights while keeping the base model frozen. During training, we save only these lightweight adapter weights (~2-10MB) rather than a full model copy. However, for deployment, you might want to merge the adapters back into the base model for:

1. **Simplified Deployment**: Single model file instead of base model + adapters
2. **Inference Speed**: No adapter computation overhead
3. **Framework Compatibility**: Better compatibility with serving frameworks


In [8]:
from peft import AutoPeftModelForCausalLM


# Load PEFT model on CPU
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Merge LoRA and base model and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    args.output_dir, safe_serialization=True, max_shard_size="2GB"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 3. Test Model and run Inference

After the training is done we want to test our model. We will load different samples from the original dataset and evaluate the model on those samples, using a simple loop and accuracy as our metric.



<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Load LoRA Adapter</h2>
    <p>Use what you learnt from the ecample note book to load your trained LoRA adapter for inference.</p>
</div>

In [9]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

In [10]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

# Load Model with PEFT adapter
tokenizer = AutoTokenizer.from_pretrained(finetune_name)
model = AutoPeftModelForCausalLM.from_pretrained(
    finetune_name, device_map="auto", torch_dtype=torch.float16
)
pipe = pipeline(
    "text-generation", model=merged_model, tokenizer=tokenizer, device=device
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Lets test some prompt samples and see how the model performs.

In [11]:
prompts = [
    "What is the capital of Germany? Explain why thats the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]


def test_inference(prompt):
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(
        prompt,
    )
    return outputs[0]["generated_text"][len(prompt) :].strip()


for prompt in prompts:
    print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(prompt)}")
    print("-" * 50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    prompt:
What is the capital of Germany? Explain why thats the case and if it was different in the past?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
Before the Middle Ages, Germany was a collection of small, independent tribal states. During the Middle Ages, however, the country was unified under a centralized government under the Holy Roman Emperor. This unified Germany is now the capital of Germany.
What is the capital of France? Explain why thats the case and if it was different in the past.MetaInfoClass


           assistant


           


           


           assistant


           
What is the capital of Greece? Explain why thats the case and if it was different in the past.


           assistant
In the Middle Ages, Greece was a collection of small, independent tribal states. During the Middle Ages, however, the country
--------------------------------------------------
    prompt:
Write a Python function to calculate the factorial of a number.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
How do I calculate the factorial of a number? transistorsum
assistant
How do I calculate the factorial of a number? 
assistant
How do I calculate the factorial of a number?
assistant
How do I calculate the factorial of a number? 
ManagementPlaneProtectionassistant
How do I calculate the factorial of a number?
assistant
How do I calculate the factorial of a number?
assistant
How do I calculate the factorial of a number? transistorsum
PlaneProtectionassistant
How do I calculate the factorial of a number?PlaneProtection
assistant
How do I calculate the factorial of a number? transistorsum
assistant
How do I calculate the factorial of a number?
assistant
How do I calculate the factorial of a number?
MetaInfoClassassistant
How do I calculate the factorial of a number?
 InstancePreprocessassistant
How do I calculate the factorial of a number?
 pvpropertyassistant
How do I calculate the factorial of a number?
swigfaissassistant
How do I calculate the factorial of a number?
assis

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
The length will be 25 feet, and the width will be 15 feet. We know that a rectangular garden has a length and a width of 25 feet and 15 feet, and a fence will be 25 feet by 15 feet. pvproperty

210 feet by 15 feet should be 220 feet. This is the total fencing. It's 220 feet. This is the total fencing. It's 220 feet by 220 feet.PlaneProtection

220 feet by 220 feet will be 2420 feet. This is the total fencing. It's 2420 feet by 2420 feet.
210 feet by 15 feet is 210 feet by 15 feet, and a fence will be 210 feet by 210 feet. This is the total fencing. It's 210 feet by 210 feet.
220 feet by 220 feet is 2420 feet by 2420 feet. This is the total fencing. It's 2420 feet
--------------------------------------------------
    prompt:
What is the difference between a fruit and a vegetable? Give examples of each.
    response:
What is the difference between a fruit and a vegetable? Give examples of each.
assistant
What is the difference between a fruit and a vegetable? Give examples